In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import plotnine as p9
import liana as li
import muon as mu
import anndata as ad
import matplotlib.pyplot as plt
import squidpy as sq
from liana.method._pipe_utils._common import _get_props
from liana.method.sp._utils import _add_complexes_to_var
from liana._logging import _logg
from liana.method._pipe_utils import prep_check_adata, assert_covered
from scipy.sparse import csr_matrix

c:\Users\athee\anaconda3\envs\liana_new\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
c:\Users\athee\anaconda3\envs\liana_new\lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.


In [2]:
adata = sc.read_h5ad("232_temp.h5ad")

MemoryError: Unable to allocate 923. MiB for an array with shape (242059315,) and data type int32

In [ ]:
subset_adata = adata[np.random.choice(adata.shape[0], size=1000, replace=False)]

## Method 1

In [ ]:
from docs.source.notebooks.inflow_score_function_edit import SpatialInflow
inflow = SpatialInflow()

OSError: [Errno 22] Invalid argument

In [17]:
global_res1, lrdata1 = inflow(subset_adata,  
                                  groupby='ist', # cell type columns
                                  resource_name='cellchatdb',
                                  use_raw=False)
                                  
lrdata1.shape

c:\Users\athee\anaconda3\envs\liana_new\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
h:\Other computers\My Laptop (1)\Google Drive\PhD_project\Spatial_and_CCC_with_Daniel\liana-py\docs\source\notebooks\inflow_score_test.py:209: RuntimeWarning: divide by zero encountered in divide


(1000, 102)

## Method 2 (original)

In [18]:
from liana.method.sp._inflow import inflow

In [19]:
global_res2, lrdata2 = li.mt.inflow(subset_adata,  
                                  groupby='ist', # cell type columns
                                  resource_name='cellchatdb',
                                  use_raw=False)
                                  
lrdata2.shape

c:\Users\athee\anaconda3\envs\liana_new\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
H:\Other computers\My Laptop (1)\Google Drive\PhD_project\Spatial_and_CCC_with_Daniel\liana-py\liana\method\sp\_inflow\inflow.py:184: RuntimeWarning: invalid value encountered in divide


(1000, 102)

In [36]:
lrdata1.X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 163 stored elements and shape (1000, 102)>

In [ ]:
lrdata2.X the 

ArrayView([[0., 0., 0., ..., 0., 0., 0.],
           [0., 0., 0., ..., 0., 0., 0.],
           [0., 0., 0., ..., 0., 0., 0.],
           ...,
           [0., 0., 0., ..., 0., 0., 0.],
           [0., 0., 0., ..., 0., 0., 0.],
           [0., 0., 0., ..., 0., 0., 0.]], shape=(1000, 102))

In [38]:
# Convert sparse matrix to dense array
dense1 = lrdata1.X.toarray()

# Convert ArrayView to NumPy array if needed
dense2 = np.array(lrdata2.X)

# Check if they are exactly the same
are_equal = np.array_equal(dense1, dense2)

print("Are the matrices exactly the same?", are_equal)

Are the matrices exactly the same? False


In [ ]:
# Convert sparse matrix to dense
dense_array = lrdata1.X.toarray()
# Create DataFrame
df = pd.DataFrame(dense_array, index=lrdata1.obs_names, columns=lrdata1.var_names)
df2 = pd.DataFrame(lrdata2.X, index=lrdata2.obs_names, columns=lrdata2.var_names)


In [1]:
import random

In [47]:
for _ in range(5):  # Run 5 random checks
    random_col = random.choice(df.columns)
    sum_df1 = df[random_col].sum()
    sum_df2 = df2[random_col].sum()
    
    print(f"\nRandomly selected column: {random_col}")
    print(f"Sum in df: {sum_df1}")
    print(f"Sum in df2: {sum_df2}")
    
    if sum_df1 == sum_df2:
        print("✅ The column sums are equal.")
    else:
        print("❌ The column sums are different.")



Randomly selected column: epi2^LGALS9^CD44
Sum in df: 10.246716709415542
Sum in df2: 10.246716930188747
❌ The column sums are different.

Randomly selected column: fib1^COL6A3^SDC4
Sum in df: 8.537839371028003
Sum in df2: 8.537838926045538
❌ The column sums are different.

Randomly selected column: epi4^LAMB3^ITGA6_ITGB4
Sum in df: 6.004914821827783
Sum in df2: 6.004914605945714
❌ The column sums are different.

Randomly selected column: epi2^LAMC2^ITGA6_ITGB1
Sum in df: 4.074009557091686
Sum in df2: 4.074009672853549
❌ The column sums are different.

Randomly selected column: epi2^EFNB1^EPHB4
Sum in df: 8.822450854475727
Sum in df2: 8.822450948113953
❌ The column sums are different.


In [34]:
global_res2.sort_values('name').tail() # NOTE: should correspond to the most commonly expressed (by cell type)

,name,value
599,fib2^MIF^CD44_CD74^epi4,0.028394
701,fib2^MIF^CD44_CD74^fib1,0.000000
803,fib2^MIF^CD44_CD74^fib2,0.000000
905,fib2^MIF^CD44_CD74^mye1,0.000000
1007,fib2^MIF^CD44_CD74^mye2,0.000000


In [35]:
global_res1.sort_values('name').tail() # NOTE: should correspond to the most commonly expressed (by cell type)


,name,value
599,fib2^MIF^CD44_CD74^epi4,0.028394
701,fib2^MIF^CD44_CD74^fib1,0.000000
803,fib2^MIF^CD44_CD74^fib2,0.000000
905,fib2^MIF^CD44_CD74^mye1,0.000000
1007,fib2^MIF^CD44_CD74^mye2,0.000000
